# Huấn luyện AI cho cảm biến nhiệt độ + IR phát hiện lửa

**Mục tiêu:** Học từ các phiên đo **bình thường** bằng Isolation Forest, rồi xuất `model.joblib` cho FastAPI.

**Quan trọng:** Đây là AI phát hiện bất thường, **không phải** bộ phát hiện/chữa cháy đã kiểm định. ESP32 phải có cảnh báo khẩn độc lập, không phụ thuộc API. Không dùng ngọn lửa thật hay khí ga để tạo dữ liệu.

In [ ]:
%pip -q install pandas numpy scikit-learn joblib

## Bước 1. Tải dữ liệu

CSV **chỉ chứa dữ liệu môi trường bình thường đã được xác nhận**, 1 mẫu/giây, gồm `session_id,temperature,infrared`; `timestamp_ms` là tùy chọn.
- `session_id`: mã phiên đo, ví dụ `normal_01`
- `temperature`: nhiệt độ °C
- `infrared`: **1 khi cảm biến thấy tín hiệu lửa**, 0 khi không thấy (chuẩn hóa trước ở ESP32)

Nên có ít nhất 3 phiên đo độc lập. File `example_normal_SIMULATED.csv` chỉ để thử kỹ thuật, không đại diện tình huống thực tế.

In [ ]:
from google.colab import files
import io
import pandas as pd

up = files.upload()
assert len(up) == 1, "Chỉ tải một CSV ở bước này"
filename, data = next(iter(up.items()))
df = pd.read_csv(io.BytesIO(data))
if "flame" in df and "infrared" not in df:
    df = df.rename(columns={"flame":"infrared"})
required = {"session_id", "temperature", "infrared"}
assert required.issubset(df.columns), f"Thiếu cột: {required - set(df.columns)}"
df = df.dropna(subset=["temperature", "infrared", "session_id"]).copy()
df["temperature"] = pd.to_numeric(df["temperature"], errors="raise")
df["infrared"] = pd.to_numeric(df["infrared"], errors="raise")
assert df["infrared"].isin([0,1]).all(), "infrared phải là 0/1"
assert df["infrared"].eq(0).all(), "File huấn luyện normal chỉ được có infrared=0"
assert df["temperature"].between(-40,125).all(), "Nhiệt độ ngoài phạm vi hợp lý: kiểm tra lại cảm biến"
print(df.groupby("session_id").size())
display(df.head())

## Bước 2. Chia thành các cửa sổ 10 giây và trích xuất đặc trưng

Một cửa sổ có 10 mẫu cách nhau khoảng 1 giây; không nối dữ liệu giữa các phiên đo. Các đặc trưng: trung bình nhiệt độ, nhiệt độ cao nhất, nhiệt độ thay đổi, tỷ lệ cảm biến IR báo lửa.

In [ ]:
import numpy as np
from sklearn.model_selection import GroupShuffleSplit

WINDOW = 10
FEATURE_NAMES = ["temp_mean", "temp_max", "temp_change", "flame_ratio"]
rows = []
for session, part in df.groupby("session_id", sort=False):
    if "timestamp_ms" in part.columns:
        part = part.sort_values("timestamp_ms")
    x = part[["temperature", "infrared"]].to_numpy(dtype=np.float32)
    for start in range(0, len(x) - WINDOW + 1, WINDOW):
        w = x[start:start+WINDOW]
        temp, ir = w[:,0], w[:,1]
        rows.append({"session_id": session, "temp_mean":float(temp.mean()),
                     "temp_max":float(temp.max()), "temp_change":float(temp[-1]-temp[0]),
                     "flame_ratio":float(ir.mean())})
features_df = pd.DataFrame(rows)
assert len(features_df) >= 20, "Cần ít nhất 20 cửa sổ; hãy thu thêm dữ liệu"
assert features_df.session_id.nunique() >= 3, "Cần >=3 phiên đo để đánh giá theo phiên độc lập"
display(features_df.head())
print("Số cửa sổ:",len(features_df))

## Bước 3. Huấn luyện và kiểm thử normal chưa thấy

Tách theo **session_id**, tránh trộn các đoạn của cùng phiên vào cả train và test. `IsolationForest` không tạo ra xác suất cháy; nó trả điểm bất thường. Trên tập normal độc lập, ta đo được **tỷ lệ báo động giả**, không thể kết luận độ nhạy phát hiện cháy.

In [ ]:
from sklearn.ensemble import IsolationForest

split = GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=42)
idx_train, idx_test = next(split.split(features_df, groups=features_df.session_id))
train = features_df.iloc[idx_train]
test = features_df.iloc[idx_test]
X_train = train[FEATURE_NAMES].to_numpy(dtype=np.float32)
X_test = test[FEATURE_NAMES].to_numpy(dtype=np.float32)
model = IsolationForest(n_estimators=150, contamination=0.05, random_state=42)
model.fit(X_train)
y_test = model.predict(X_test)  # 1 = bình thường, -1 = bất thường
print("Số cửa sổ train/test:",len(X_train),len(X_test))
print("Tỷ lệ báo động giả trên normal test: {:.1%}".format((y_test == -1).mean()))
print("Điểm mẫu (không phải xác suất):",model.decision_function(X_test[:5]))

## Bước 4. Xuất mô hình

Tải file về máy, đặt chung thư mục với `api.py`. Chỉ dùng file joblib do chính bạn tạo. Phiên bản scikit-learn ở server cần tương thích phiên bản lúc huấn luyện.

In [ ]:
import joblib, sklearn
from google.colab import files

artifact = {"model":model, "window_size":WINDOW, "feature_names":FEATURE_NAMES,
            "sample_period_ms":1000, "sklearn_version":sklearn.__version__}
joblib.dump(artifact, "model.joblib")
with open("training_versions.txt", "w", encoding="utf-8") as f:
    f.write("scikit-learn==" + sklearn.__version__ + "\n")
print("Xuất thành công model.joblib, sklearn:",sklearn.__version__)
files.download("model.joblib")
files.download("training_versions.txt")